In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()


# =========================================================
# 5. COMPILE
# =========================================================

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 6. THREAD 001
# =========================================================

config_001 = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n===================================")
print("FIRST RUN - THREAD 001")
print("===================================")

result_1 = graph.invoke(
    {
        "count": 0
    },
    config=config_001
)

print("Result:", result_1)


# =========================================================
# 7. CHECK THREAD 001 STATE
# =========================================================

state_001 = graph.get_state(
    config_001
)

print("\nTHREAD 001 CURRENT STATE")

print(
    "Thread ID:",
    state_001.config["configurable"]["thread_id"]
)

print(
    "Values:",
    state_001.values
)

print(
    "Checkpoint ID:",
    state_001.config["configurable"]["checkpoint_id"]
)


# =========================================================
# 8. RUN THREAD 001 AGAIN
# =========================================================

print("\n===================================")
print("SECOND RUN - SAME THREAD 001")
print("===================================")

result_2 = graph.invoke(
    {},
    config=config_001
)

print("Result:", result_2)


# =========================================================
# 9. CHECK THREAD 001 AGAIN
# =========================================================

state_001_after = graph.get_state(
    config_001
)

print("\nTHREAD 001 AFTER SECOND INVOKE")

print(
    "Thread ID:",
    state_001_after.config[
        "configurable"
    ]["thread_id"]
)

print(
    "Values:",
    state_001_after.values
)

print(
    "New Checkpoint ID:",
    state_001_after.config[
        "configurable"
    ]["checkpoint_id"]
)


# =========================================================
# 10. THREAD 001 HISTORY
# =========================================================

print("\n===================================")
print("THREAD 001 HISTORY")
print("===================================")

history_001 = list(
    graph.get_state_history(
        config_001
    )
)

print(
    "Total Checkpoints:",
    len(history_001)
)

for state in history_001:

    print("\n-----------------------------")

    print(
        "Checkpoint:",
        state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 11. THREAD 002
# =========================================================

config_002 = {
    "configurable": {
        "thread_id": "thread-002"
    }
}


print("\n===================================")
print("THREAD 002")
print("===================================")

result_3 = graph.invoke(
    {
        "count": 100
    },
    config=config_002
)

print("Result:", result_3)


# =========================================================
# 12. THREAD 002 STATE
# =========================================================

state_002 = graph.get_state(
    config_002
)

print("\nTHREAD 002 CURRENT STATE")

print(
    "Thread ID:",
    state_002.config[
        "configurable"
    ]["thread_id"]
)

print(
    "Values:",
    state_002.values
)

print(
    "Checkpoint ID:",
    state_002.config[
        "configurable"
    ]["checkpoint_id"]
)


# =========================================================
# 13. FINAL COMPARISON
# =========================================================

print("\n===================================")
print("FINAL COMPARISON")
print("===================================")

print(
    "\nTHREAD 001:"
)

print(
    "State:",
    state_001_after.values
)

print(
    "Checkpoint:",
    state_001_after.config[
        "configurable"
    ]["checkpoint_id"]
)


print(
    "\nTHREAD 002:"
)

print(
    "State:",
    state_002.values
)

print(
    "Checkpoint:",
    state_002.config[
        "configurable"
    ]["checkpoint_id"]
)